# Classificação de Cogumelos com K-Nearest Neighbors (KNN)

**Objetivo:** Prever se um cogumelo é **venenoso (p)** ou **comestível (e)** com base em suas características físicas, utilizando o algoritmo KNN.

**Dataset:** Mushroom Dataset (UCI).

---

### O que é o KNN?

O **K-Nearest Neighbors** é um algoritmo de classificação simples. Para classificar um novo ponto, ele calcula a **distância** até todos os outros exemplos do dataset, seleciona os **K mais próximos** e retorna a classe que aparece **com mais frequência** entre eles.

## Importação das bibliotecas

In [19]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## 1. Carregamento dos dados

In [20]:
df = pd.read_csv('../data.csv')

print(f'Colunas lidas: {list(df.columns)}\n')
df

Colunas lidas: ['class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']



,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,e,k,s,n,f,n,a,c,b,y,...,s,o,o,p,o,o,p,b,c,l
8120,e,x,s,n,f,n,a,c,b,y,...,s,o,o,p,n,o,p,b,v,l
8121,e,f,s,n,f,n,a,c,b,n,...,s,o,o,p,o,o,p,b,c,l
8122,p,k,y,n,f,y,f,c,n,b,...,k,w,w,p,w,o,e,w,v,l


---
## 2. Limpeza e Preparação dos Dados

In [21]:

FEATURES = [ col for col in df.columns if col != 'class' ]
TARGET = 'class'

le_features = {}
for col in FEATURES:
    le_col = LabelEncoder()
    df[col] = le_col.fit_transform(df[col].astype(str))
    le_features[col] = le_col

print(f'Serão no total {len(FEATURES)} features utilizadas')

Serão no total 22 features utilizadas


### 2.1 Distribuição do Target

In [22]:
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
 
df = df.dropna(subset=FEATURES + [TARGET]).copy()

display(df.head())

print('Distribuição do target:')
print(df[TARGET].value_counts())

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,5,2,4,1,6,1,0,1,4,...,2,7,7,0,2,1,4,2,3,5
1,e,5,2,9,1,0,1,0,0,4,...,2,7,7,0,2,1,4,3,2,1
2,e,0,2,8,1,3,1,0,0,5,...,2,7,7,0,2,1,4,3,2,3
3,p,5,3,8,1,6,1,0,1,5,...,2,7,7,0,2,1,4,2,3,5
4,e,5,2,3,0,5,1,1,0,4,...,2,7,7,0,2,1,0,3,0,1


Distribuição do target:
class
e    4208
p    3916
Name: count, dtype: int64


---
## 3. Escolha e Preparação das Features

In [23]:
le = LabelEncoder()

y_all = le.fit_transform(df[TARGET])
X_all = df[FEATURES].values

print(f'Classes: {list(le.classes_)}')
print(f'Features: {FEATURES}')

Classes: ['e', 'p']
Features: ['cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']


---
## 4. Separação em Treino e Teste

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print(f'Treino: {len(X_train)/len(X_all)*100:.0f}% = {len(X_train)} amostras')
print(f'Teste: {len(X_test)/len(X_all)*100:.0f}% = {len(X_test)} amostras')

Treino: 80% = 6499 amostras
Teste: 20% = 1625 amostras


---
## 5. Treinamento com o KNN

In [25]:
model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=3, weights='distance'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

---
## 6. Avaliação do Modelo

In [26]:
print(f"Acurácia: {accuracy_score(y_test, y_pred):.2%}")

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred))


Acurácia: 100.00%

Relatório de classificação:
              precision    recall  f1-score   support

           e       1.00      1.00      1.00       842
           p       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625

Matriz de confusão:
[[842   0]
 [  0 783]]


In [27]:
print('Exemplos de Previsões:')

sample = pd.DataFrame({
    'Real'     : le.inverse_transform(y_test),
    'Previsto' : le.inverse_transform(y_pred),
}).reset_index(drop=True)

print(sample.head(10).to_string(index=False))

Exemplos de Previsões:
Real Previsto
   p        p
   p        p
   e        e
   p        p
   p        p
   e        e
   e        e
   e        e
   e        e
   p        p
